In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if project_root not in sys.path:
    sys.path.append(project_root)
print(f"Project root added: {project_root}")

In [ ]:
import multiprocessing as mp
if mp.get_start_method(allow_none=True) is None:
    mp.set_start_method('spawn')

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message='.*input_shape.*')
warnings.filterwarnings('ignore', message='.*structure of.*inputs.*')

import os, time, gc
from types import SimpleNamespace

import numpy as np
import pandas as pd
import time as time_module
from scipy.stats import t
from scipy.special import kv, gamma

import jax, jax.numpy as jnp

import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras import backend as K

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import OneHotEncoder

import optuna
import plotly.io as pio

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import random

In [ ]:
np_f32 = np.float32
jnp_f32 = jnp.float32
dtype_basis = np.float32

jax.config.update("jax_enable_x64", False)

pio.renderers.default = "notebook"
warnings.filterwarnings("ignore", category=UserWarning)

os.environ.update({"TF_CPP_MIN_LOG_LEVEL": "2"})
optuna.logging.set_verbosity(optuna.logging.WARNING)

os.environ.setdefault("OMP_NUM_THREADS", "12")
os.environ.setdefault("MKL_NUM_THREADS", "12")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "12")

def init_hardware(dtype: str = "float32"):
    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    strategy = (tf.distribute.MirroredStrategy() if len(gpus) > 1 else tf.distribute.get_strategy())
    mixed_precision.set_global_policy(dtype)
    return strategy

strategy = init_hardware(dtype="float32")

In [ ]:
from IPython.display import display, Javascript

def save_notebook():
    display(Javascript('IPython.notebook.save_checkpoint()'))
    current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
    print(f"Notebook saved at {current_time}")

In [ ]:
from spherical_deepkriging.basis_functions.wendland.wenland import wendland
from spherical_deepkriging.basis_functions.mrts.mrts import mrts0

from spherical_deepkriging.models.deep_kriging import DeepKrigingTrainer, DeepKrigingDefaultTrainer
from spherical_deepkriging.configs import DeepKrigingModelConfig, DeepKrigingDefaultConfig
from spherical_deepkriging.models.universal_kriging import UniversalKriging

from rpy2.robjects.conversion import localconverter
from spherical_deepkriging.basis_functions.mrts_sphere.sphere import mrts_sphere, numpy2ri_converter

In [ ]:
def simulate_data(num_sample, outlier_ratio, outlier_multiplier, seed, scenario='E1'):
    rng = np.random.default_rng(seed)
    c = 1.5 if scenario == 'E1' else 0.5
    sigma_y_sq = 1.0
    a, b = 2.0, 1.0

    phi = rng.uniform(0, 2 * np.pi, num_sample)
    theta = np.arccos(rng.uniform(-1, 1, num_sample))
    lat_rad = np.pi/2 - theta
    lon_rad = phi - np.pi

    x_c = np.cos(lat_rad) * np.cos(lon_rad)
    y_c = np.cos(lat_rad) * np.sin(lon_rad)
    z_c = np.sin(lat_rad)
    coords = np.column_stack([x_c, y_c, z_c]).astype(np.float32)

    base_wind = 5.0
    westerlies_north = 18.0 * np.exp(-((lat_rad - np.pi/4)**2) / 0.05)
    westerlies_south = 22.0 * np.exp(-((lat_rad + np.pi/4)**2) / 0.04)
    doldrums = -4.0 * np.exp(-(lat_rad**2) / 0.01)
    mountain_block = np.where((lon_rad > 0.0) & (lon_rad < 1.0) & (lat_rad > 0.1) & (lat_rad < 1.0), -12.0, 0.0)
    mean_trend = base_wind + westerlies_north + westerlies_south + doldrums + mountain_block

    num_anomalies = 60
    anomaly_lats = np.arcsin(rng.uniform(-1, 1, num_anomalies))
    anomaly_lons = rng.uniform(-np.pi, np.pi, num_anomalies)
    ax = np.cos(anomaly_lats) * np.cos(anomaly_lons)
    ay = np.cos(anomaly_lats) * np.sin(anomaly_lons)
    az = np.sin(anomaly_lats)
    anomaly_effect = np.zeros(num_sample)
    for i in range(num_anomalies):
        sq_dists = (x_c - ax[i])**2 + (y_c - ay[i])**2 + (z_c - az[i])**2
        amplitude = rng.uniform(-10.0, 18.0)
        radius = rng.uniform(0.005, 0.03)
        anomaly_effect += amplitude * np.exp(-sq_dists / radius)

    mean_trend += anomaly_effect
    mean_trend = np.maximum(mean_trend, 0.5)

    dist_matrix = np.arccos(np.clip(coords @ coords.T, -1.0, 1.0))
    cov_matrix = (sigma_y_sq * np.exp(-dist_matrix / c)).astype(np.float32)
    cov_matrix += np.float32(1e-3) * np.eye(num_sample, dtype=np.float32)
    try:
        L = np.linalg.cholesky(cov_matrix)
    except np.linalg.LinAlgError:
        cov_matrix += np.float32(1e-2) * np.eye(num_sample, dtype=np.float32)
        eigenvals, eigenvecs = np.linalg.eigh(cov_matrix)
        eigenvals = np.maximum(eigenvals, 1e-6)
        L = eigenvecs @ np.diag(np.sqrt(eigenvals))

    z_std = rng.standard_normal(num_sample).astype(np.float32)
    xi = (L @ z_std).astype(np.float32)
    variance_scaler = 1.0 + 1.5 * np.exp(-((np.abs(lat_rad) - np.pi/4)**2) / 0.1)
    xi = xi * variance_scaler

    term_const = 1.0 - (1.0 / (9.0 * a))
    term_xi = xi * np.sqrt(1.0 / (9.0 * a))
    gamma_noise = (a / b) * np.power(np.maximum(term_const + term_xi, 0.0), 3)
    gamma_noise = gamma_noise - np.mean(gamma_noise)

    z_final = mean_trend + gamma_noise
    z_final = np.maximum(z_final, 0.0)
    idx_outliers = rng.choice(num_sample, size=int(num_sample * outlier_ratio), replace=False)
    z_final[idx_outliers] *= outlier_multiplier

    print(f"\n=== Scenario {scenario} (Non-stationary Windspeed + Outliers {outlier_ratio*100:.1f}% x{outlier_multiplier}) ===")
    print(f"Simulate Data | z mean: {np.mean(z_final):.4f} (\u00b1{np.std(z_final) / np.sqrt(num_sample):.4f}), Variance: {np.var(z_final, ddof=0):.4f}, Range: [{np.min(z_final):.4f}, {np.max(z_final):.4f}]")

    lat_deg = np.rad2deg(lat_rad).astype(np.float32)
    lon_deg = np.rad2deg(lon_rad).astype(np.float32)

    del coords, dist_matrix, cov_matrix, L, z_std, xi, gamma_noise, mean_trend
    del westerlies_north, westerlies_south, doldrums, mountain_block, anomaly_effect
    gc.collect()

    return pd.DataFrame({"longitude": lon_deg, "latitude": lat_deg, "z": z_final})

In [ ]:
def data_preprocessing(data_frame):
    data = data_frame.copy()
    numeric_cols = ["longitude", "latitude", "z"]
    data[numeric_cols] = data[numeric_cols].apply(pd.to_numeric, errors="coerce")
    data.dropna(subset=numeric_cols, inplace=True)
    lon, lat = data["longitude"].to_numpy(), data["latitude"].to_numpy()
    norm_lon = (lon - lon.min()) / (lon.max() - lon.min())
    norm_lat = (lat - lat.min()) / (lat.max() - lat.min())
    location_data = np.column_stack([lat, lon]).astype("float32")
    location_data_norm = np.column_stack([norm_lon, norm_lat]).astype("float32")
    y_combined = data['z'].to_numpy().astype("float32")[:, None]
    categorical_data = None
    return location_data, location_data_norm, categorical_data, y_combined


def precompute_wendland(location, num_basis):
    parts = []
    for nb in num_basis:
        grid = np.column_stack(np.meshgrid(
            np.linspace(0, 1, int(np.sqrt(nb)), dtype=np_f32),
            np.linspace(0, 1, int(np.sqrt(nb)), dtype=np_f32),
        )).reshape(-1, 2).astype(np_f32)
        theta = np_f32(2.5)/np_f32(np.sqrt(nb))
        parts.append(wendland(location, grid, theta=theta, k=2))
        del grid
        gc.collect()
    return np.hstack(parts).astype(dtype_basis, copy=False)


def precompute_max_mrts(distance_type, location_data, knot_num, order_max, knot=None):
    if knot is None:
        idx_knot = np.random.choice(location_data.shape[0], knot_num, replace=False)
        knot = location_data[idx_knot].astype(np_f32)
    else:
        idx_knot = None

    if distance_type == "sphere":
        with localconverter(numpy2ri_converter):
            res_r = mrts_sphere(knot, order_max, location_data.astype(np_f32))
        res_dict = dict(zip(res_r.names(), res_r))
        phi = np.asarray(res_dict["mrts"], dtype=dtype_basis)
    else:
        phi = np.asarray(
            mrts0(jnp.asarray(knot, dtype=jnp_f32), k=order_max,
                  x=jnp.asarray(location_data, dtype=jnp_f32)), dtype=dtype_basis
        )
    return phi, idx_knot, knot


def prepare_data(categorical_data, basis, y_combined, seed, split_ratio=(0.8, 0.1, 0.1)):
    idx_all = np.arange(basis.shape[0])
    train_ratio, val_ratio, test_ratio = split_ratio
    train_val_x1, test_x1 = train_test_split(
        idx_all, train_size=train_ratio+val_ratio, random_state=seed)
    train_x1, val_x1 = train_test_split(
        train_val_x1, train_size=train_ratio/(train_ratio+val_ratio), random_state=seed)
    X_train_cont = basis[train_x1]
    X_val_cont   = basis[val_x1]
    X_test_cont  = basis[test_x1]
    y_train, y_val, y_test = y_combined[train_x1], y_combined[val_x1], y_combined[test_x1]
    flatten_y = lambda t: t.reshape(-1).astype(np_f32, copy=False)
    y_train_flat, y_val_flat, y_test_flat = map(flatten_y, [y_train, y_val, y_test])
    flatten_x = lambda c: c.reshape(-1, basis.shape[1]).astype(np_f32)
    X_train_flat, X_val_flat, X_test_flat = map(flatten_x, [X_train_cont, X_val_cont, X_test_cont])
    if categorical_data is None:
        zv = lambda n: np.zeros((n, 0), dtype=np_f32)
        X_train_cat, X_val_cat, X_test_cat = map(zv, [len(X_train_flat), len(X_val_flat), len(X_test_flat)])
    else:
        cat_train = categorical_data.reshape(-1, 1)[train_x1]
        cat_val   = categorical_data.reshape(-1, 1)[val_x1]
        cat_test  = categorical_data.reshape(-1, 1)[test_x1]
        OHE = OneHotEncoder(sparse_output=False, dtype=np_f32)
        X_train_cat = OHE.fit_transform(cat_train).astype(np_f32)
        X_val_cat   = OHE.transform(cat_val).astype(np_f32)
        X_test_cat  = OHE.transform(cat_test).astype(np_f32)
    return (X_train_flat, X_train_cat, y_train_flat,
            X_val_flat,   X_val_cat,   y_val_flat,
            X_test_flat,  X_test_cat,  y_test_flat)

In [ ]:
def cleanup_tf_session():
    K.clear_session()
    gc.collect()
    try:
        tf.keras.backend.clear_session()
    except:
        pass


def train_eval(name_model, epochs, batch_size, loss, dropout_rate,
               X_train, X_train_cat, y_train,
               X_val, X_val_cat, y_val,
               X_test, X_test_cat, y_test):

    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    if name_model in ["OLS_wendland", "OLS_sphere"]:
        t0 = time.time()
        model = LinearRegression().fit(X_train, y_train)
        val_loss = float(mean_squared_error(y_val, model.predict(X_val)))
        y_pred = model.predict(X_test).astype(np_f32).reshape(-1)
        training_time = time.time() - t0
        metrics = {
            "Model": name_model, "Val_loss": float(val_loss),
            "MSPE": float(mean_squared_error(y_test, y_pred)),
            "RMSE": float(np.sqrt(float(mean_squared_error(y_test, y_pred)))),
            "MAE":  float(mean_absolute_error(y_test, y_pred)),
            "R2":   float(r2_score(y_test, y_pred)),
            "Time": float(training_time),
        }
        return metrics, model

    elif name_model == "DeepKriging_wendland":
        config = DeepKrigingDefaultConfig(
            input_dim=X_train.shape[1], output_type='continuous',
            optimizer=Adam(learning_rate=1e-3), loss=loss,
            epochs=epochs, batch_size=batch_size, verbose=0
        )

    elif name_model in ["DeepKriging_mrts", "DeepKriging_sphere", "DeepKriging_sphere_Huber"]:
        # clipnorm=1.0 prevents gradient explosion from heavy-tailed sphere basis columns
        _opt = Adam(learning_rate=5e-3, clipnorm=1.0) if "sphere" in name_model else Adam(learning_rate=5e-3)
        config = DeepKrigingModelConfig(
            input_dim=X_train.shape[1], output_type='continuous',
            hidden_layers=[1024, 512, 256, 128, 64], activation='relu',
            dropout_rate=dropout_rate, optimizer=_opt,
            loss=loss, metrics=['mae'], epochs=epochs, batch_size=batch_size,
            patience=40, verbose=0
        )

    t0 = time.time()
    with strategy.scope():
        model = DeepKrigingDefaultTrainer(config) if name_model == "DeepKriging_wendland" else DeepKrigingTrainer(config)
        model.model.compile(optimizer=config.optimizer, loss=config.loss, metrics=config.metrics)

    checkpoint_path = f"best_{name_model}_{time.time_ns()}.weights.h5"
    if name_model == "DeepKriging_wendland":
        callbacks = [tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path, monitor="val_loss", mode="min",
            save_best_only=True, save_weights_only=True, verbose=0)]
    else:
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=checkpoint_path, monitor="val_loss", mode="min",
                save_best_only=True, save_weights_only=True, verbose=0),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=config.patience,
                restore_best_weights=True, verbose=0),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5,
                patience=max(5, config.patience // 2), min_lr=1e-6, verbose=0)
        ]

    train_dataset = tf.data.Dataset.from_tensor_slices(
        ((X_train, X_train_cat), y_train)).batch(config.batch_size).prefetch(tf.data.AUTOTUNE)
    val_dataset = tf.data.Dataset.from_tensor_slices(
        ((X_val, X_val_cat), y_val)).batch(config.batch_size).prefetch(tf.data.AUTOTUNE)

    history = model.model.fit(
        train_dataset, validation_data=val_dataset,
        epochs=epochs, callbacks=callbacks, verbose=0)

    if os.path.exists(checkpoint_path):
        model.model.load_weights(checkpoint_path)
        os.remove(checkpoint_path)

    val_loss = float(np.min(history.history["val_loss"]))
    y_pred = model.model.predict([X_test, X_test_cat], verbose=0).reshape(-1).astype(np_f32)
    training_time = time.time() - t0

    metrics = {
        "Model": name_model, "Val_loss": float(val_loss),
        "MSPE": float(mean_squared_error(y_test, y_pred)),
        "RMSE": float(np.sqrt(float(mean_squared_error(y_test, y_pred)))),
        "MAE":  float(mean_absolute_error(y_test, y_pred)),
        "R2":   float(r2_score(y_test, y_pred)),
        "Time": float(training_time),
    }
    del train_dataset, val_dataset
    gc.collect()
    return metrics, model

In [ ]:
def plot_robinson(ax, longitude, latitude, value, vmin, vmax, title):
    ax.set_global()
    ax.add_feature(cfeature.LAND,      facecolor="white", alpha=0.3)
    ax.add_feature(cfeature.OCEAN,     facecolor="white", alpha=0.3)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.3, alpha=0.5)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.2, alpha=0.5)
    sc = ax.scatter(longitude, latitude, c=value,
                    cmap=mcolors.LinearSegmentedColormap.from_list(
                        "teal-yellow-red", ["#00AAAA", "#FFFFBB", "#FF3333"], N=256),
                    s=10, transform=ccrs.PlateCarree(), vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10, pad=3)
    return sc


def create_subplot_robinson(fig, position, locations, values, vmin, vmax, title,
                             plot_type='prediction', cbar_label=None):
    ax = fig.add_subplot(*position, projection=ccrs.Robinson())
    cmap = (mcolors.LinearSegmentedColormap.from_list(
                "blue-white-red", ["#2166AC", "#F7F7F7", "#B2182B"], N=256)
            if plot_type == 'residual' else
            mcolors.LinearSegmentedColormap.from_list(
                "teal-yellow-red", ["#00AAAA", "#FFFFBB", "#FF3333"], N=256))
    ax.set_global()
    ax.add_feature(cfeature.LAND,      facecolor="white", alpha=0.3)
    ax.add_feature(cfeature.OCEAN,     facecolor="white", alpha=0.3)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.3, alpha=0.5)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.2, alpha=0.5)
    sc = ax.scatter(locations['longitude'], locations['latitude'], c=values,
                    cmap=cmap, s=10, transform=ccrs.PlateCarree(), vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10, pad=3)
    cbar = plt.colorbar(sc, ax=ax, orientation='horizontal', fraction=0.046, pad=0.04, shrink=0.8)
    cbar.set_label(cbar_label or ("Residual" if plot_type == 'residual' else "Prediction Value"), fontsize=10)
    cbar.ax.tick_params(labelsize=7)
    return ax, sc


def visualize_comparison(dataframe, models_dict, basis_dict, y_combined, seed,
                          model_list=None, experiment_info=None):
    if model_list is None:
        model_list = ['DeepKriging_sphere', 'DeepKriging_sphere_Huber', 'UniversalKriging']
    idx_all = np.arange(len(y_combined))
    _, test_idx = train_test_split(idx_all, train_size=0.9, random_state=seed)
    y_test = y_combined[test_idx].reshape(-1)
    test_locations = dataframe.iloc[test_idx]

    predictions = {}
    for model_name in model_list:
        if model_name not in models_dict or models_dict[model_name] is None:
            continue
        model  = models_dict[model_name]
        X_test = basis_dict[model_name][test_idx]
        if "DeepKriging" in model_name:
            X_test_cat = np.zeros((len(X_test), 0), dtype=np.float32)
            y_pred = model.model.predict([X_test, X_test_cat], verbose=0).reshape(-1)
        elif model_name == "UniversalKriging":
            coords_test = dataframe[['longitude', 'latitude']].iloc[test_idx].values.astype(np.float32)
            y_pred = model.predict(coords_test, X_test, return_centered=False)
        else:
            y_pred = model.predict(X_test).reshape(-1)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        order = models_dict.get(f"{model_name}_order", "")
        predictions[model_name] = {'values': y_pred, 'rmse': rmse, 'order': order}

    all_vals = np.concatenate([dataframe['z'].values] + [p['values'] for p in predictions.values()])
    vmin, vmax = np.percentile(all_vals, 2), np.percentile(all_vals, 98)

    fig1 = plt.figure(figsize=(16, 14))
    create_subplot_robinson(fig1, (2, 2, 1), dataframe, dataframe['z'], vmin, vmax,
                            f'Real Data (n={len(dataframe)})')
    for i, mn in enumerate(model_list):
        if mn in predictions:
            p = predictions[mn]
            dn = mn.replace('DeepKriging_sphere','DK_S').replace('_Huber','_H').replace('UniversalKriging','UK')
            create_subplot_robinson(fig1, (2, 2, i+2), test_locations, p['values'], vmin, vmax,
                                    f"{dn} (order={p['order']}) | RMSE={p['rmse']:.4f}")
    plt.suptitle("Prediction Comparison — Sphere ColNorm Fix", fontsize=16, fontweight='bold', y=0.84)
    plt.tight_layout(rect=[0, 0.02, 1, 0.94])
    plt.show(); plt.close(fig1)

    fig2 = plt.figure(figsize=(18, 6))
    residuals_map = {k: (y_test - predictions[k]['values']) for k in model_list if k in predictions}
    vmax_abs = max(np.max(np.abs(r)) for r in residuals_map.values())
    for i, mn in enumerate(model_list):
        if mn in predictions:
            dn = mn.replace('DeepKriging_sphere','DK_S').replace('_Huber','_H').replace('UniversalKriging','UK')
            create_subplot_robinson(fig2, (1, 3, i+1), test_locations, residuals_map[mn],
                                    -vmax_abs, vmax_abs,
                                    f"{dn} Residuals (order={predictions[mn]['order']})",
                                    plot_type='residual')
    plt.suptitle("Residuals Comparison — Sphere ColNorm Fix", fontsize=16, fontweight='bold', y=0.84)
    plt.tight_layout(rect=[0, 0.02, 1, 0.94])
    plt.show(); plt.close(fig2)
    return predictions, test_idx

In [ ]:
# ── Experiment parameters ─────────────────────────────────────────────────────
seed = 50
OUTLIER_RATIO       = 0.025
OUTLIER_MULTIPLIER  = 5
epochs = 500
batch_size = 256
num_sample = 2500
huber_delta = 1.345

num_basis = [10**2, 19**2, 37**2]
knot_num  = 1400
order_max = 1400

# All models (including dk_sphere) use the same original candidates
base_orders = [10, 50, 100, 150, 200, 1000]

repeat_experiment = 50

print(f"FIX: max_Phi_sphere will be column-normalized to unit L2 norm after precompute")
print(f"dk_sphere candidates : {base_orders}  (restored to original)")
print(f"repeats              : {repeat_experiment}")

In [ ]:
experiment_results = {
    model: {"MSPE": [], "RMSE": [], "MAE": [], "R2": []}
    for model in ["OLS_wendland", "OLS_sphere", "DeepKriging_wendland",
                  "DeepKriging_mrts", "DeepKriging_sphere", "DeepKriging_sphere_Huber",
                  "UniversalKriging"]
}

for repeat in range(repeat_experiment):
    seed = seed + repeat

    print(f"\n{'='*80}")
    print(f"Repeat {repeat+1}/{repeat_experiment}, Seed={seed}")
    print(f"{'='*80}")

    best_orders = {}

    dataframe = simulate_data(num_sample=num_sample, outlier_ratio=OUTLIER_RATIO, outlier_multiplier=OUTLIER_MULTIPLIER, seed=seed)
    location_data, location_data_norm, categorical_data, y_combined = data_preprocessing(dataframe)

    # Compute sphere basis
    max_Phi_sphere, idx_knot, knot = precompute_max_mrts(
        "sphere", location_data, knot_num, order_max, knot=None)
    max_Phi_sphere = max_Phi_sphere.astype(dtype_basis, copy=False)

    # ── FIX: per-column StandardScaler (fit on train split only) ────────────
    from sklearn.preprocessing import StandardScaler as _SS
    _tv_idx, _ = train_test_split(np.arange(max_Phi_sphere.shape[0]), train_size=0.9, random_state=seed)
    _tr_idx, _ = train_test_split(_tv_idx, train_size=8/9, random_state=seed)
    _sphere_scaler = _SS()
    _sphere_scaler.fit(max_Phi_sphere[_tr_idx])
    max_Phi_sphere = _sphere_scaler.transform(max_Phi_sphere).astype(dtype_basis)
    print(f"Sphere basis after StandardScaler: mean={max_Phi_sphere.mean():.4f}, std={max_Phi_sphere.std():.4f}")
    # ─────────────────────────────────────────────────────────────────────────

    # Compute euclidean MRTS basis (same knots as sphere)
    max_Phi_mrts, idx_knot_mrts, knot_mrts = precompute_max_mrts(
        "mrts", location_data, knot_num, order_max, knot=location_data[idx_knot])
    max_Phi_mrts = max_Phi_mrts.astype(dtype_basis, copy=False)

    # Compute Wendland basis
    Phi_wendland = precompute_wendland(location_data_norm, num_basis)


    # =========================================================================
    # TUNING PHASE
    # =========================================================================

    # OLS_sphere
    Best_val_OLS_S, Best_order_OLS_S, Results_order_OLS_S = float('inf'), None, []
    print("\nTuning OLS_sphere")
    for order in base_orders:
        Phi_sphere = max_Phi_sphere[:, :order].astype(np_f32)
        parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed)
        metrics, model = train_eval("OLS_sphere", None, None, None, None, *parts)
        Results_order_OLS_S.append({'order': order, 'val_loss': metrics["Val_loss"], 'mspe': metrics["MSPE"]})
        if metrics["Val_loss"] < Best_val_OLS_S:
            Best_val_OLS_S = metrics["Val_loss"]; Best_order_OLS_S = order
        del Phi_sphere, parts, model; gc.collect()
    print(f"   {'Order':<10} {'Val Loss':<12} {'Test MSE':<12}")
    for res in Results_order_OLS_S:
        mk = " *" if res['order'] == Best_order_OLS_S else ""
        print(f"   {res['order']:<10} {res['val_loss']:<12.4f} {res['mspe']:<12.4f}{mk}")
    print(f"   Best order: {Best_order_OLS_S}")
    best_orders['OLS_sphere'] = Best_order_OLS_S


    # DeepKriging_mrts
    Best_val_DK_mrts, Best_order_DK_mrts, Results_order_DK_mrts = float('inf'), None, []
    print("\nTuning DeepKriging_mrts")
    for order in base_orders:
        Phi_mrts = max_Phi_mrts[:, :order].astype(np_f32)
        parts = prepare_data(categorical_data, Phi_mrts, y_combined, seed)
        with strategy.scope():
            metrics, model = train_eval("DeepKriging_mrts", epochs, batch_size, "mse", 0.01, *parts)
        Results_order_DK_mrts.append({'order': order, 'val_loss': metrics["Val_loss"], 'mspe': metrics["MSPE"]})
        if metrics["Val_loss"] < Best_val_DK_mrts:
            Best_val_DK_mrts = metrics["Val_loss"]; Best_order_DK_mrts = order
        del Phi_mrts, parts; cleanup_tf_session()
    print(f"   {'Order':<10} {'Val Loss':<12} {'Test MSE':<12}")
    for res in Results_order_DK_mrts:
        mk = " *" if res['order'] == Best_order_DK_mrts else ""
        print(f"   {res['order']:<10} {res['val_loss']:<12.4f} {res['mspe']:<12.4f}{mk}")
    print(f"   Best order: {Best_order_DK_mrts}")
    best_orders['DeepKriging_mrts'] = Best_order_DK_mrts


    # DeepKriging_sphere  ← now with col-normalized basis + original candidates
    Best_val_DK_S, Best_order_DK_S, Results_order_DK_S = float('inf'), None, []
    print(f"\nTuning DeepKriging_sphere (candidates={base_orders}, col-norm fix applied)")
    for order in base_orders:
        Phi_sphere = max_Phi_sphere[:, :order].astype(np_f32)
        parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed)
        with strategy.scope():
            metrics, model = train_eval("DeepKriging_sphere", epochs, batch_size, "mse", 0.01, *parts)
        Results_order_DK_S.append({'order': order, 'val_loss': metrics["Val_loss"], 'mspe': metrics["MSPE"]})
        if metrics["Val_loss"] < Best_val_DK_S:
            Best_val_DK_S = metrics["Val_loss"]; Best_order_DK_S = order
        del Phi_sphere, parts; cleanup_tf_session()
    print(f"   {'Order':<10} {'Val Loss':<12} {'Test MSE':<12}")
    for res in Results_order_DK_S:
        mk = " *" if res['order'] == Best_order_DK_S else ""
        print(f"   {res['order']:<10} {res['val_loss']:<12.4f} {res['mspe']:<12.4f}{mk}")
    print(f"   Best order: {Best_order_DK_S}")
    best_orders['DeepKriging_sphere'] = Best_order_DK_S


    # DeepKriging_sphere_Huber
    Best_val_DK_S_H, Best_order_DK_S_H, Results_order_DK_S_H = float('inf'), None, []
    print(f"\nTuning DeepKriging_sphere_Huber (candidates={base_orders})")
    for order in base_orders:
        Phi_sphere = max_Phi_sphere[:, :order].astype(np_f32)
        parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed)
        with strategy.scope():
            metrics, model = train_eval("DeepKriging_sphere_Huber", epochs, batch_size,
                                        Huber(delta=huber_delta), 0.01, *parts)
        Results_order_DK_S_H.append({'order': order, 'val_loss': metrics["Val_loss"], 'mspe': metrics["MSPE"]})
        if metrics["Val_loss"] < Best_val_DK_S_H:
            Best_val_DK_S_H = metrics["Val_loss"]; Best_order_DK_S_H = order
        del Phi_sphere, parts; cleanup_tf_session()
    print(f"   {'Order':<10} {'Val Loss':<12} {'Test MSE':<12}")
    for res in Results_order_DK_S_H:
        mk = " *" if res['order'] == Best_order_DK_S_H else ""
        print(f"   {res['order']:<10} {res['val_loss']:<12.4f} {res['mspe']:<12.4f}{mk}")
    print(f"   Best order: {Best_order_DK_S_H}")
    best_orders['DeepKriging_sphere_Huber'] = Best_order_DK_S_H


    # UniversalKriging
    Best_val_UK, Best_order_UK, Results_order_UK = float('inf'), None, []
    print("\nTuning UniversalKriging")
    for order in base_orders:
        Phi_sphere = max_Phi_sphere[:, :order].astype(np_f32)
        idx_all = np.arange(Phi_sphere.shape[0])
        train_val_idx, test_idx = train_test_split(idx_all, train_size=0.9, random_state=seed)
        train_idx, val_idx = train_test_split(train_val_idx, train_size=8/9, random_state=seed)
        coords_train, coords_val = location_data[train_idx], location_data[val_idx]
        phi_train, phi_val       = Phi_sphere[train_idx], Phi_sphere[val_idx]
        y_tr, y_vl = y_combined[train_idx].flatten(), y_combined[val_idx].flatten()
        uk_model = UniversalKriging(num_neighbors=30, cov_function='exponential')
        uk_model.fit(coords_train, phi_train, y_tr, center_y=True)
        y_pred_val = uk_model.predict(coords_val, phi_val, return_centered=True)
        val_loss   = mean_squared_error(y_vl - uk_model.y_mean, y_pred_val)
        coords_test, phi_test = location_data[test_idx], Phi_sphere[test_idx]
        y_te = y_combined[test_idx].flatten()
        y_pred_test = uk_model.predict(coords_test, phi_test, return_centered=False)
        test_mse = mean_squared_error(y_te, y_pred_test)
        Results_order_UK.append({'order': order, 'val_loss': val_loss, 'mspe': test_mse})
        if val_loss < Best_val_UK:
            Best_val_UK = val_loss; Best_order_UK = order
        uk_model.cleanup()
        del uk_model, Phi_sphere, coords_train, coords_val, coords_test
        del phi_train, phi_val, phi_test, y_tr, y_vl, y_te; gc.collect()
    print(f"   {'Order':<10} {'Val Loss':<12} {'Test MSE':<12}")
    for res in Results_order_UK:
        mk = " *" if res['order'] == Best_order_UK else ""
        print(f"   {res['order']:<10} {res['val_loss']:<12.4f} {res['mspe']:<12.4f}{mk}")
    print(f"   Best order: {Best_order_UK}")
    best_orders['UniversalKriging'] = Best_order_UK
    gc.collect()


    # =========================================================================
    # EVALUATION PHASE
    # =========================================================================
    Record = {}

    # OLS_wendland
    parts = prepare_data(categorical_data, Phi_wendland, y_combined, seed)
    metric, model = train_eval("OLS_wendland", None, None, None, None, *parts)
    Record["OLS_wendland"] = {
        "MSPE": metric["MSPE"], "RMSE": metric["RMSE"], "MAE": metric["MAE"], "R2": metric["R2"],
        "Time": metric["Time"], "Param": "--", "model": model if repeat == 0 else None}
    if repeat != 0: del model
    del parts; gc.collect()

    # OLS_sphere
    Phi_sphere = max_Phi_sphere[:, :best_orders['OLS_sphere']].astype(np_f32)
    parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed)
    metric, model = train_eval("OLS_sphere", None, None, None, None, *parts)
    Record["OLS_sphere"] = {
        "MSPE": metric["MSPE"], "RMSE": metric["RMSE"], "MAE": metric["MAE"], "R2": metric["R2"],
        "Time": metric["Time"], "Param": best_orders['OLS_sphere'], "model": model if repeat == 0 else None}
    if repeat != 0: del model
    del Phi_sphere, parts; gc.collect()

    # DeepKriging_wendland
    parts = prepare_data(categorical_data, Phi_wendland, y_combined, seed)
    with strategy.scope():
        metric, model = train_eval("DeepKriging_wendland", epochs, batch_size, "mse", 0.01, *parts)
    Record["DeepKriging_wendland"] = {
        "MSPE": metric["MSPE"], "RMSE": metric["RMSE"], "MAE": metric["MAE"], "R2": metric["R2"],
        "Time": metric["Time"], "Param": "--", "model": model if repeat == 0 else None}
    if repeat != 0: del model; cleanup_tf_session()
    del parts; gc.collect()

    # DeepKriging_mrts
    Phi_mrts = max_Phi_mrts[:, :best_orders['DeepKriging_mrts']].astype(np_f32)
    parts = prepare_data(categorical_data, Phi_mrts, y_combined, seed)
    with strategy.scope():
        metric, model = train_eval("DeepKriging_mrts", epochs, batch_size, "mse", 0.01, *parts)
    Record["DeepKriging_mrts"] = {
        "MSPE": metric["MSPE"], "RMSE": metric["RMSE"], "MAE": metric["MAE"], "R2": metric["R2"],
        "Time": metric["Time"], "Param": best_orders['DeepKriging_mrts'], "model": model if repeat == 0 else None}
    if repeat != 0: del model; cleanup_tf_session()
    del Phi_mrts, parts; gc.collect()

    # DeepKriging_sphere  (col-norm fix)
    Phi_sphere = max_Phi_sphere[:, :best_orders['DeepKriging_sphere']].astype(np_f32)
    parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed)
    with strategy.scope():
        metric, model = train_eval("DeepKriging_sphere", epochs, batch_size, "mse", 0.01, *parts)
    Record["DeepKriging_sphere"] = {
        "MSPE": metric["MSPE"], "RMSE": metric["RMSE"], "MAE": metric["MAE"], "R2": metric["R2"],
        "Time": metric["Time"], "Param": best_orders['DeepKriging_sphere'], "model": model if repeat == 0 else None}
    if repeat != 0: del model; cleanup_tf_session()
    del Phi_sphere, parts; gc.collect()

    # DeepKriging_sphere_Huber
    Phi_sphere = max_Phi_sphere[:, :best_orders['DeepKriging_sphere_Huber']].astype(np_f32)
    parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed)
    with strategy.scope():
        metric, model = train_eval("DeepKriging_sphere_Huber", epochs, batch_size,
                                   Huber(delta=huber_delta), 0.01, *parts)
    Record["DeepKriging_sphere_Huber"] = {
        "MSPE": metric["MSPE"], "RMSE": metric["RMSE"], "MAE": metric["MAE"], "R2": metric["R2"],
        "Time": metric["Time"], "Param": best_orders['DeepKriging_sphere_Huber'], "model": model if repeat == 0 else None}
    if repeat != 0: del model; cleanup_tf_session()
    del Phi_sphere, parts; gc.collect()

    # UniversalKriging
    t0 = time.time()
    Phi_sphere = max_Phi_sphere[:, :best_orders['UniversalKriging']].astype(np_f32)
    idx_all = np.arange(Phi_sphere.shape[0])
    train_val_idx, test_idx = train_test_split(idx_all, train_size=0.9, random_state=seed)
    train_idx, _ = train_test_split(train_val_idx, train_size=8/9, random_state=seed)
    coords_train, coords_test = location_data[train_idx], location_data[test_idx]
    phi_train, phi_test = Phi_sphere[train_idx], Phi_sphere[test_idx]
    y_tr, y_te = y_combined[train_idx].flatten(), y_combined[test_idx].flatten()
    uk_model = UniversalKriging(num_neighbors=30, cov_function='exponential')
    uk_model.fit(coords_train, phi_train, y_tr, center_y=True)
    y_pred_test = uk_model.predict(coords_test, phi_test, return_centered=False)
    Record["UniversalKriging"] = {
        "MSPE": mean_squared_error(y_te, y_pred_test),
        "RMSE": np.sqrt(mean_squared_error(y_te, y_pred_test)),
        "MAE":  mean_absolute_error(y_te, y_pred_test),
        "R2":   r2_score(y_te, y_pred_test),
        "Time": time.time() - t0,
        "Param": best_orders['UniversalKriging'],
        "model": uk_model if repeat == 0 else None}
    if repeat != 0: uk_model.cleanup(); del uk_model
    del Phi_sphere, coords_train, coords_test, phi_train, phi_test, y_tr, y_te; gc.collect()


    # Print repeat results
    result_table = []
    for mn in ["OLS_wendland", "OLS_sphere", "DeepKriging_wendland", "DeepKriging_mrts",
               "DeepKriging_sphere", "DeepKriging_sphere_Huber", "UniversalKriging"]:
        result_table.append({
            "Model": mn, "Param": Record[mn]["Param"],
            "MSPE": f"{Record[mn]['MSPE']:.4f}", "RMSE": f"{Record[mn]['RMSE']:.4f}",
            "MAE":  f"{Record[mn]['MAE']:.4f}",  "R2":   f"{Record[mn]['R2']:.4f}",
            "Time": f"{Record[mn]['Time']:.2f}s"
        })
    df_res = pd.DataFrame(result_table)
    print("\n", df_res.to_markdown(index=False, tablefmt="github"), sep="")

    # Visualize on first repeat only
    if repeat == 0:
        visualized_model = {
            "DeepKriging_sphere":       Record["DeepKriging_sphere"]["model"],
            "DeepKriging_sphere_order": Record["DeepKriging_sphere"]["Param"],
            "DeepKriging_sphere_Huber":       Record["DeepKriging_sphere_Huber"]["model"],
            "DeepKriging_sphere_Huber_order": Record["DeepKriging_sphere_Huber"]["Param"],
            "UniversalKriging":       Record["UniversalKriging"]["model"],
            "UniversalKriging_order": Record["UniversalKriging"]["Param"],
        }
        visualized_basis = {
            "DeepKriging_sphere":       max_Phi_sphere[:, :best_orders['DeepKriging_sphere']],
            "DeepKriging_sphere_Huber": max_Phi_sphere[:, :best_orders['DeepKriging_sphere_Huber']],
            "UniversalKriging":         max_Phi_sphere[:, :best_orders['UniversalKriging']],
        }
        visualize_comparison(
            dataframe, visualized_model, visualized_basis, y_combined, seed,
            model_list=['DeepKriging_sphere', 'DeepKriging_sphere_Huber', 'UniversalKriging'])
        cleanup_tf_session()

    # Aggregate
    for mn in ["OLS_wendland", "OLS_sphere", "DeepKriging_wendland", "DeepKriging_mrts",
               "DeepKriging_sphere", "DeepKriging_sphere_Huber", "UniversalKriging"]:
        experiment_results[mn]["MSPE"].append(Record[mn]["MSPE"])
        experiment_results[mn]["RMSE"].append(Record[mn]["RMSE"])
        experiment_results[mn]["MAE"].append(Record[mn]["MAE"])
        experiment_results[mn]["R2"].append(Record[mn]["R2"])

    del Phi_wendland, max_Phi_sphere, max_Phi_mrts, dataframe, location_data, location_data_norm
    cleanup_tf_session(); gc.collect()

    # Checkpoint
    _rows = []
    for _m, _v in experiment_results.items():
        if _v['RMSE']:
            _rows.append({'Model': _m, 'Repeat': repeat+1,
                          'MSPE': _v['MSPE'][-1], 'RMSE': _v['RMSE'][-1],
                          'MAE':  _v['MAE'][-1],  'R2':   _v['R2'][-1]})
    _ckpt_path = 'results_sphere_stdscaler_outliers_50reps_checkpoint.csv'
    _ckpt_df = pd.DataFrame(_rows)
    if os.path.exists(_ckpt_path):
        _ckpt_df.to_csv(_ckpt_path, mode='a', header=False, index=False)
    else:
        _ckpt_df.to_csv(_ckpt_path, index=False)
    print(f"Checkpoint saved -> {_ckpt_path}")

    # notebook auto-save removed — caused kernel crash when nbconvert holds the file lock

    print(f"\nCompleted Repeat {repeat+1}/{repeat_experiment}")

In [ ]:
print("\n" + "="*80)
print("Summary — Outliers + Sphere StdScaler+ClipNorm Fix")
print("="*80)
print(f"Fix applied: StandardScaler on sphere basis + Adam clipnorm=1.0 for sphere")
print(f"dk_sphere candidates: {base_orders}")
print("="*80)

avg_results = []
for mn in ["OLS_wendland", "OLS_sphere", "DeepKriging_wendland", "DeepKriging_mrts",
           "DeepKriging_sphere", "DeepKriging_sphere_Huber", "UniversalKriging"]:
    m = experiment_results[mn]
    avg_results.append({
        "Model": mn,
        "MSPE": f"{np.mean(m['MSPE']):.2f}\u00b1{np.std(m['MSPE']):.2f}",
        "RMSE": f"{np.mean(m['RMSE']):.2f}\u00b1{np.std(m['RMSE']):.2f}",
        "MAE":  f"{np.mean(m['MAE']):.2f}\u00b1{np.std(m['MAE']):.2f}",
        "R2":   f"{np.mean(m['R2']):.2f}\u00b1{np.std(m['R2']):.2f}",
    })

df_avg = pd.DataFrame(avg_results)
print("\n", df_avg.to_markdown(index=False, tablefmt="github"), sep="")

if avg_results:
    best_model = min(avg_results, key=lambda x: float(x['RMSE'].split('\u00b1')[0]))
    print(f"\nBest Model: {best_model['Model']} (RMSE: {best_model['RMSE']})")